# KKBOX 표본 고객 100,000명 선정

## 목적
- 프로젝트에서 사용할 고객 100,000명을 선정한다.
- 팀원 모두 동일한 고객을 기준으로 전처리, EDA, 피처 엔지니어링을 진행한다.
- msno를 공통 고객 식별자로 사용한다.

## 예측 문제 정의
- **예측 시점(관찰 종료일)**: 2017-02-28 (모든 고객 동일)
- **대상 고객**: 2017년 3월 멤버십 만료 고객 (`train_v2.csv`)
- **예측 대상**: 만료 후 30일 이내 재구독하지 않음 → `is_churn = 1`
- **입력 데이터**: 2017-02-28 이하에 기록된 거래·로그만 사용

## 표본 추출 기준
- 모집단: `train_v2.csv` 고객 중 `members_v3`에 2017-02-28 이후 가입한 것으로 확인되지 않은 고객
  - `members_v3`에 없는 고객은 가입일을 확인할 수 없으므로 표본에 남김
  - 거래·로그가 없는 고객도 표본에 남김 → 피처 생성 시 0 또는 결측으로 처리
  - 3월 로그/거래 존재 여부로 고르면 2/28 이후 정보로 표본을 고르는 것이 되어 이탈 고객이 덜 뽑힘
- 추출 방법: `is_churn` 기준 층화 추출
- 표본 크기: 100,000명 / 난수 고정값: `random_state=42`

## 사용 데이터
| 파일 | 용도 | 날짜 필터 |
|---|---|---|
| train_v2.csv | 정답 라벨 | - |
| members_v3.csv | 가입일 확인 및 회원 정보 (left join) | 가입일 ≤ 20170228인 행 사용 |
| transactions.csv + transactions_v2.csv | 결제 이력 | transaction_date ≤ 20170228 |
| user_logs.csv + user_logs_v2.csv | 청취 로그 | LOG_START ≤ date ≤ 20170228 |


# 0. 설정

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

RAW = Path("../data/raw")
SAVE = Path("../data/sampled")
SAVE.mkdir(parents=True, exist_ok=True)

N_SAMPLE = 100_000
SEED = 42
OBS_END = 20170228      # 관찰 종료일 (이 날짜 이하 데이터만 사용)
LOG_START = 20161201    # 로그 시작일. 기간을 늘리면 처리 시간이 증가 (None이면 전체 기간)
CHUNK = 5_000_000       # 메모리 부족하면 줄이기

def valid_yyyymmdd(s, column):
    """8자리 YYYYMMDD 날짜인지 확인하고 정수값으로 돌려준다."""
    raw = s.astype("string")
    parsed = pd.to_datetime(raw, format="%Y%m%d", errors="coerce")
    bad = parsed.isna() | ~raw.str.fullmatch(r"\d{8}").fillna(False)
    if bad.any():
        raise ValueError(f"{column}: 유효하지 않은 날짜 {bad.sum():,}건 (예: {raw[bad].head(3).tolist()})")
    return parsed.dt.strftime("%Y%m%d").astype("int64")

def read_filtered(path, keep_msno, date_col, max_date, min_date=None, chunksize=CHUNK):
    """큰 CSV를 청크로 읽으면서 선정 고객 + 날짜 범위에 해당하는 행만 남긴다."""
    parts, total = [], 0
    for i, chunk in enumerate(pd.read_csv(path, chunksize=chunksize)):
        total += len(chunk)
        chunk = chunk.loc[chunk["msno"].isin(keep_msno)].copy()
        if not chunk.empty:
            dates = valid_yyyymmdd(chunk[date_col], f"{path.name}.{date_col}")
            mask = dates.le(max_date)
            if min_date is not None:
                mask &= dates.ge(min_date)
            parts.append(chunk.loc[mask])
        print(f"  {path.name}: {total:,}행 읽음", end="\r")
    print()
    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=pd.read_csv(path, nrows=0).columns)
    print(f"  {path.name}: {len(out):,}행 남음")
    return out

def write_filtered_logs(paths, keep_msno, output, min_date, max_date, chunksize=CHUNK):
    """로그를 청크별로 파일에 기록한다. 전체 로그를 메모리에 쌓지 않는다."""
    first = True
    count = 0
    seen_customers = set()
    for path in paths:
        for chunk in pd.read_csv(path, chunksize=chunksize):
            chunk = chunk.loc[chunk["msno"].isin(keep_msno)].copy()
            if chunk.empty:
                continue
            dates = valid_yyyymmdd(chunk["date"], f"{path.name}.date")
            lo = min_date if min_date is not None else 0
            kept = chunk.loc[dates.between(lo, max_date)]
            if kept.empty:
                continue
            kept.to_csv(output, mode="w" if first else "a", header=first, index=False)
            first = False
            count += len(kept)
            seen_customers.update(kept["msno"])
        print(f"  {path.name}: 처리 완료 / 현재 {count:,}행")
    if first:
        pd.read_csv(paths[0], nrows=0).to_csv(output, index=False)
    return count, seen_customers


# 1. 라벨 불러오기 및 10만 명 층화 추출

In [2]:
target = (
    pd.read_csv(RAW / "train_v2.csv")
    [["msno", "is_churn"]]
    .dropna(subset=["is_churn"])
    .drop_duplicates(subset="msno")
)

print(f"train_v2 고객 수: {len(target):,}명")

# 예측 시점에 아직 가입하지 않은 것으로 확인된 고객만 모집단에서 제외한다.
# members_v3에 행이 없는 고객은 가입일을 확인할 수 없으므로 그대로 둔다.
members_all = pd.read_csv(RAW / "members_v3.csv").drop_duplicates(subset="msno")
members_all = members_all[members_all["msno"].isin(set(target["msno"]))]  # train_v2 고객만 검사
reg = valid_yyyymmdd(members_all["registration_init_time"], "registration_init_time")
future_msno = set(members_all.loc[reg.gt(OBS_END), "msno"])
excluded = target["msno"].isin(future_msno)
print(f"2/28 이후 가입자로 확인되어 제외: {excluded.sum():,}명")
target = target.loc[~excluded].reset_index(drop=True)
print(f"층화 추출 모집단: {len(target):,}명")
print(target["is_churn"].value_counts())
print(target["is_churn"].value_counts(normalize=True))

train_v2 고객 수: 970,960명
2/28 이후 가입자로 확인되어 제외: 1,057명
층화 추출 모집단: 969,903명
is_churn
0    883083
1     86820
Name: count, dtype: int64
is_churn
0    0.910486
1    0.089514
Name: proportion, dtype: float64


In [3]:
if len(target) < N_SAMPLE:
    raise ValueError(f"고객이 {len(target):,}명이라 {N_SAMPLE:,}명을 추출할 수 없습니다.")

target_100k, _ = train_test_split(
    target,
    train_size=N_SAMPLE,
    stratify=target["is_churn"],
    random_state=SEED,
)
target_100k = target_100k.reset_index(drop=True)
selected_msno = set(target_100k["msno"])

print(f"선정 고객 수: {len(selected_msno):,}명")
print(target_100k["is_churn"].value_counts())
print(target_100k["is_churn"].value_counts(normalize=True))

선정 고객 수: 100,000명
is_churn
0    91049
1     8951
Name: count, dtype: int64
is_churn
0    0.91049
1    0.08951
Name: proportion, dtype: float64


# 2. 회원 정보 (없는 고객은 결측으로 남김)

In [4]:
members_100k = (
    members_all[members_all["msno"].isin(selected_msno)]
    .reset_index(drop=True)
)
# 모집단에서 제외한 고객이 회원 정보에 다시 들어오지 않았는지 확인한다.
assert not members_100k["msno"].isin(future_msno).any()
print(f"회원 정보 있는 고객: {len(members_100k):,}명 / 없는 고객: {N_SAMPLE - len(members_100k):,}명")
del members_all


회원 정보 있는 고객: 88,648명 / 없는 고객: 11,352명


# 3. 결제 이력 (transaction_date ≤ 2017-02-28)

In [5]:
transactions_100k = pd.concat([
    read_filtered(RAW / "transactions.csv", selected_msno, "transaction_date", OBS_END),
    read_filtered(RAW / "transactions_v2.csv", selected_msno, "transaction_date", OBS_END),
], ignore_index=True).drop_duplicates().reset_index(drop=True)

print(f"결제: {len(transactions_100k):,}행 / {transactions_100k['msno'].nunique():,}명")

  transactions.csv: 21,547,746행 읽음
  transactions.csv: 1,560,801행 남음
  transactions_v2.csv: 1,431,009행 읽음
  transactions_v2.csv: 17,602행 남음
결제: 1,578,144행 / 99,862명


# 4. 청취 로그 (LOG_START ≤ date ≤ 2017-02-28)

- `user_logs.csv`는 약 30GB이므로 청크 단위로 필터링해 바로 CSV에 기록한다.
- 로그가 없는 고객도 `train_100k.csv`에는 남는다.
- 같은 고객·같은 날짜에 여러 행이 존재할 수 있으므로 여기서 삭제하지 않는다. 피처 생성 시 고객별로 필요한 값들을 집계한다.


In [6]:
LOG_OUTPUT = SAVE / "user_logs_100k.csv"
log_count, log_customers = write_filtered_logs(
    [RAW / "user_logs.csv", RAW / "user_logs_v2.csv"],
    selected_msno, LOG_OUTPUT, LOG_START, OBS_END,
)
print(f"로그: {log_count:,}행 / {len(log_customers):,}명")


  user_logs.csv: 처리 완료 / 현재 3,890,410행
  user_logs_v2.csv: 처리 완료 / 현재 3,890,410행
로그: 3,890,410행 / 81,719명


# 5. 검증

In [7]:
# 1) 날짜 형식과 관찰 종료일 검사 (잘못된 날짜는 예외 발생)
tx_dates = valid_yyyymmdd(transactions_100k["transaction_date"], "transaction_date")
assert tx_dates.le(OBS_END).all()
assert transactions_100k["msno"].isin(selected_msno).all()
assert members_100k["msno"].isin(selected_msno).all()
assert valid_yyyymmdd(members_100k["registration_init_time"], "registration_init_time").le(OBS_END).all()

# 로그 파일도 청크별로 검사하므로 전체 로그를 다시 메모리에 올리지 않는다.
log_min, log_max, checked_rows = None, None, 0
same_day_extra_in_chunks = 0
for chunk in pd.read_csv(LOG_OUTPUT, chunksize=CHUNK):
    dates = valid_yyyymmdd(chunk["date"], "user_logs_100k.date")
    assert dates.between(LOG_START if LOG_START is not None else 0, OBS_END).all()
    assert chunk["msno"].isin(selected_msno).all()
    checked_rows += len(chunk)
    same_day_extra_in_chunks += chunk.duplicated(["msno", "date"]).sum()
    if not dates.empty:
        log_min = min(log_min, dates.min()) if log_min is not None else dates.min()
        log_max = max(log_max, dates.max()) if log_max is not None else dates.max()
assert checked_rows == log_count
assert len(target_100k) == N_SAMPLE and target_100k["msno"].is_unique
print("검증 통과 ✅")
print("transaction_date 범위:", tx_dates.min(), "~", tx_dates.max())
print("log date 범위       :", log_min, "~", log_max)
print(f"동일 고객·날짜의 추가 로그 행(청크 내부만 계산, 최소 {same_day_extra_in_chunks:,}행): 피처 집계 시 유지")

# 2) 기록 없는 고객도 표본에 남아 있는지 확인
cov = target_100k.assign(
    has_member=target_100k["msno"].isin(set(members_100k["msno"])),
    has_tx=target_100k["msno"].isin(set(transactions_100k["msno"])),
    has_log=target_100k["msno"].isin(log_customers),
)
for col in ["has_member", "has_tx", "has_log"]:
    print(f"\n[{col}]")
    print(cov.groupby(col)["is_churn"].agg(고객수="size", 이탈률="mean"))


검증 통과 ✅
transaction_date 범위: 20150101 ~ 20170228
log date 범위       : 20161201 ~ 20170228
동일 고객·날짜의 추가 로그 행(청크 내부만 계산, 최소 0행): 피처 집계 시 유지

[has_member]
              고객수       이탈률
has_member                 
False       11352  0.051445
True        88648  0.094385

[has_tx]
          고객수       이탈률
has_tx                 
False     138  0.623188
True    99862  0.088773

[has_log]
           고객수       이탈률
has_log                 
False    18281  0.069799
True     81719  0.093919


# 6. 파일 저장

In [8]:
target_100k.to_csv(SAVE / "train_100k.csv", index=False)
members_100k.to_csv(SAVE / "members_100k.csv", index=False)
transactions_100k.to_csv(SAVE / "transactions_100k.csv", index=False)
# user_logs_100k.csv는 4단계에서 청크별로 이미 저장했다.

print(f"저장 완료: {SAVE.resolve()}")
print("※ 피처 생성 시 train_100k를 기준으로 left join 할 것 (기록 없는 고객이 빠지지 않도록)")


저장 완료: C:\dev\project\2차 단위 프로젝트\2nd-fix-data\data\sampled
※ 피처 생성 시 train_100k를 기준으로 left join 할 것 (기록 없는 고객이 빠지지 않도록)
